In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.ensemble import (GradientBoostingRegressor, RandomForestRegressor,AdaBoostRegressor)
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

In [4]:
from google.colab import files
uploaded = files.upload()

Saving Vientiane_final.csv to Vientiane_final.csv


In [6]:
df = pd.read_csv("Vientiane_final.csv")
df.head()

,YEAR,MONTH,ALLSKY_SFC_SW_DWN,T2M,T2MDEW,T2MWET,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M,QV2M,WS2M,GWETTOP
0,1995,1,19.53,21.36,17.68,19.52,27.32,15.69,0.01,80.76,12.80,1.85,0.62
1,1995,1,17.67,20.32,16.36,18.34,25.94,16.46,0.04,79.18,11.79,2.12,0.62
2,1995,1,19.10,20.69,17.08,18.88,26.43,16.69,0.00,81.10,12.36,1.52,0.62
3,1995,1,19.64,17.64,12.95,15.30,23.11,13.31,0.00,75.19,9.47,2.47,0.62
4,1995,1,18.96,15.32,8.12,11.72,21.84,9.98,0.00,64.23,6.83,2.44,0.61


In [7]:

features = ["YEAR","MONTH","ALLSKY_SFC_SW_DWN","T2M","T2MDEW","T2MWET","T2M_MAX","T2M_MIN","RH2M","QV2M","WS2M","GWETTOP"]
target = "PRECTOTCORR"

In [8]:
train = df[df["YEAR"] <= 2020]
test  = df[df["YEAR"] >= 2021]
X_train, y_train = train[features], train[target]
X_test,  y_test  = test[features],  test[target]

scaler = StandardScaler()
Xtr_sc = scaler.fit_transform(X_train)
Xte_sc = scaler.transform(X_test)

In [9]:
models = {
    "Gradient Boosting": (GradientBoostingRegressor(n_estimators=200, random_state=42), False),
    "SVR":               (SVR(kernel="rbf", C=10, epsilon=0.1),                         True),
    "Random Forest":     (RandomForestRegressor(n_estimators=200, random_state=42),      False),
    "Neural Network":    (MLPRegressor(hidden_layer_sizes=(128,64,32),
                                       max_iter=500, random_state=42),                   True),
    "AdaBoost":          (AdaBoostRegressor(n_estimators=200, random_state=42),          False),
    "Decision Tree":     (DecisionTreeRegressor(random_state=42),                        False),
}

In [13]:
# ── Train & evaluate ──────────────────────────────────────────────────────────
results = {}
for name, (model, scaled) in models.items():
    Xtr = Xtr_sc if scaled else X_train.values
    Xte = Xte_sc if scaled else X_test.values
    model.fit(Xtr, y_train)
    preds = np.clip(model.predict(Xte), 0, None)
    zm = y_test == 0
    results[name] = {
        "RMSE":     round(np.sqrt(mean_squared_error(y_test, preds)), 3),
        "R2":       round(r2_score(y_test, preds), 3),
        "Zero_Acc": round((preds[zm] < 0.5).mean() * 100, 1),
    }

res_df = pd.DataFrame(results).T.reset_index().rename(columns={"index": "Model"})
print(res_df.to_string(index=False))

            Model  RMSE     R2  Zero_Acc
Gradient Boosting 4.409  0.557      82.5
              SVR 4.731  0.490      93.9
    Random Forest 4.799  0.476      82.7
   Neural Network 5.159  0.394      87.7
         AdaBoost 5.919  0.202       0.0
    Decision Tree 8.333 -0.581      89.7
